# P1 실습 — MNIST로 첫 신경망 확인하기

이 실습에서는 강의에서 배운 기본 신경망을 **MNIST 손글씨 숫자 분류 문제**에 직접 적용한다.

목표는 코드를 처음부터 작성하는 것이 아니라, 코드를 실행하면서 다음 흐름을 확인하는 것이다.

> **데이터 확인 → 입력 준비 → 모델 구성 → 훈련 → 예측 → 평가 → 오류 확인**

MNIST는 이 프로젝트의 **유도 실습용 데이터셋**이다.  
실제 P1 프로젝트에서는 같은 원리를 **Fashion-MNIST**에 적용한다.

## 1. MNIST 데이터 불러오기

MNIST는 `0`부터 `9`까지의 손글씨 숫자 이미지 데이터다.

- 훈련 이미지: 60,000장
- 테스트 이미지: 10,000장
- 이미지 한 장의 크기: `28 × 28`
- 타깃: 숫자 `0`부터 `9`

먼저 데이터를 불러오고 shape을 확인한다.

In [ ]:
from keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

print("train_images:", train_images.shape)
print("train_labels:", train_labels.shape)
print("test_images :", test_images.shape)
print("test_labels :", test_labels.shape)

### 확인

`train_images.shape`이 `(60000, 28, 28)`이라는 것은 무엇을 의미하는가?

<details>
<summary><strong>해설 보기</strong></summary>

60,000장의 이미지가 있고, 각 이미지는 `28 × 28`개의 픽셀로 이루어져 있다는 뜻이다.

</details>

## 2. 실제 이미지와 타깃 확인

shape만 보는 것보다 실제 데이터를 함께 보는 것이 중요하다.

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(8, 4))

for i in range(8):
    ax = fig.add_subplot(2, 4, i + 1)
    ax.imshow(train_images[i], cmap="gray")
    ax.set_title(f"label = {train_labels[i]}")
    ax.axis("off")

plt.tight_layout()
plt.show()

### 확인

이미지와 타깃을 함께 보면서 다음을 확인해보자.

- 입력은 무엇인가?
- 타깃은 무엇인가?
- 왜 이 문제는 다중분류 문제인가?

<details>
<summary><strong>해설 보기</strong></summary>

입력은 손글씨 숫자 이미지이고, 타깃은 `0`부터 `9`까지의 숫자다.  
10개의 범주 가운데 하나를 예측하므로 다중분류 문제다.

</details>

## 3. 완전연결층에 맞게 입력 준비

MNIST 이미지 한 장의 원래 shape은 `(28, 28)`이다.

하지만 이번에 사용할 `Dense` 층은 **완전연결층**이므로 이미지의 픽셀을 하나의 벡터로 펼쳐서 입력한다.

> `(28, 28) → (784,)`

또한 픽셀값의 범위를 `0~255`에서 `0~1`로 바꾼다.

In [ ]:
print("변환 전:", train_images.shape)
print("픽셀값 범위:", train_images.min(), "~", train_images.max())

train_images = train_images.reshape((60000, 28 * 28)).astype("float32") / 255
test_images = test_images.reshape((10000, 28 * 28)).astype("float32") / 255

print("변환 후:", train_images.shape)
print("픽셀값 범위:", train_images.min(), "~", train_images.max())

### 생각

1. reshape 전후에 이미지 한 장에 들어 있는 값의 개수는 달라졌는가?
2. 왜 `(28, 28)`을 `(784,)`로 바꾸는가?
3. `/ 255`는 무엇을 바꾸는가?

<details>
<summary><strong>해설 보기</strong></summary>

1. 값의 개수는 변하지 않는다. `28 × 28 = 784`이다.
2. 완전연결층에 784개의 픽셀값을 하나의 입력 벡터로 전달하기 위해서다.
3. 픽셀값의 범위를 `0~255`에서 `0~1`로 바꾼다.

</details>

## 4. 첫 신경망 구성

강의에서 본 기본 신경망을 그대로 사용한다.

> **784개의 입력값 → 512개의 은닉 유닛 → 10개의 출력값**

In [ ]:
import keras
from keras import layers

model = keras.Sequential([
    layers.Input(shape=(28 * 28,)),
    layers.Dense(512, activation="relu"),
    layers.Dense(10, activation="softmax"),
])

model.summary()

### 모델 구조 읽기

`model.summary()`를 보고 확인해보자.

1. 첫 번째 `Dense` 층의 출력 크기는 얼마인가?
2. 마지막 `Dense` 층의 출력이 10개인 이유는 무엇인가?
3. 강의에서 계산한 파라미터 수와 `model.summary()`의 값이 일치하는가?

<details>
<summary><strong>해설 보기</strong></summary>

- 첫 번째 `Dense` 층은 512개의 값을 출력한다.
- MNIST의 숫자 범주가 10개이므로 마지막 출력도 10개다.
- 첫 층의 파라미터 수는 `784 × 512 + 512`, 두 번째 층은 `512 × 10 + 10`이다.

</details>

## 5. 모델 훈련 준비

모델을 훈련하려면 다음 세 가지를 지정한다.

- **optimizer**: 파라미터를 어떻게 갱신할지 결정
- **loss**: 예측이 얼마나 틀렸는지 측정
- **metric**: 성능을 어떤 지표로 확인할지 결정

지금은 각 이름을 외우기보다 **코드에서 어떤 역할을 맡는지** 확인한다.

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

## 6. 모델 훈련

훈련 과정에서 일부 훈련 데이터를 **검증 데이터**로 분리한다.

훈련 정확도만 보는 것이 아니라, 학습하지 않은 검증 데이터에서도 성능이 어떻게 변하는지 함께 확인하기 위해서다.

In [ ]:
history = model.fit(
    train_images,
    train_labels,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
)

### 훈련 로그 읽기

훈련 결과에서 다음을 찾아보자.

- `loss`
- `accuracy`
- `val_loss`
- `val_accuracy`

그리고 epoch가 진행되면서 각각 어떻게 변하는지 관찰한다.

## 7. 훈련 과정을 그림으로 확인

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], marker="o", label="train")
plt.plot(history.history["val_accuracy"], marker="o", label="validation")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()
plt.show()

### 생각

훈련 정확도와 검증 정확도는 항상 같은가?

지금은 과대적합을 자세히 분석하지 않는다. 다만 다음 질문을 기억해두자.

> **훈련 데이터에서 잘 맞는 것과 새로운 데이터에서도 잘 맞는 것은 같은가?**

## 8. 실제 예측 읽기

테스트 이미지 몇 장에 대해 모델의 출력을 확인한다.

출력은 10개의 값이며, 각 값은 숫자 `0`부터 `9`에 대응한다.

In [ ]:
predictions = model.predict(test_images[:10], verbose=0)

i = 0
print("10개 출력:", predictions[i])
print("예측 숫자:", predictions[i].argmax())
print("가장 큰 출력값:", predictions[i].max())
print("실제 정답:", test_labels[i])

### 예측을 이미지와 함께 확인

In [ ]:
# 원래 모양으로 되돌려 시각화
image = test_images[0].reshape(28, 28)

plt.imshow(image, cmap="gray")
plt.title(
    f"prediction = {predictions[0].argmax()}, "
    f"label = {test_labels[0]}"
)
plt.axis("off")
plt.show()

### 확인

한 이미지의 예측 과정은 다음과 같이 읽을 수 있다.

> **이미지 → 784개 입력값 → 신경망 → 10개 출력 → 가장 큰 값의 위치 → 예측 숫자**

`argmax()`가 하는 역할을 이 흐름에서 설명해보자.

## 9. 테스트 데이터에서 최종 평가

훈련에 사용하지 않은 테스트 데이터 전체에서 모델의 성능을 확인한다.

In [ ]:
test_loss, test_acc = model.evaluate(
    test_images,
    test_labels,
    verbose=0,
)

print(f"test loss     : {test_loss:.4f}")
print(f"test accuracy : {test_acc:.4f}")

## 10. 틀린 예측 찾아보기

정확도 하나만으로는 모델이 **어떤 경우에 틀리는지** 알 수 없다.

잘못 분류된 이미지를 직접 찾아본다.

In [ ]:
all_predictions = model.predict(test_images, verbose=0).argmax(axis=1)

wrong_indices = (all_predictions != test_labels).nonzero()[0]

print("오분류 개수:", len(wrong_indices))
print("전체 테스트 이미지:", len(test_labels))

In [ ]:
fig = plt.figure(figsize=(10, 6))

for j, idx in enumerate(wrong_indices[:12]):
    ax = fig.add_subplot(3, 4, j + 1)
    ax.imshow(test_images[idx].reshape(28, 28), cmap="gray")
    ax.set_title(
        f"pred={all_predictions[idx]}, label={test_labels[idx]}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

### 판단

오분류 이미지를 몇 장 살펴보고 다음을 생각해보자.

1. 사람이 보기에도 애매한 이미지가 있는가?
2. 특정 숫자 쌍에서 혼동이 자주 보이는가?
3. 테스트 정확도 하나만 보고는 알 수 없었던 사실은 무엇인가?

이 질문들은 Fashion-MNIST 프로젝트에서 더 체계적으로 다룬다.

## 11. 실습 정리

이번 실습에서는 하나의 기본 신경망을 새로 설계하기보다, **강의에서 배운 모델이 실제 코드에서 어떻게 동작하는지 확인**했다.

핵심 흐름을 다시 정리하면 다음과 같다.

> **데이터 확인 → reshape와 정규화 → Dense 모델 구성 → 훈련 → 검증 → 예측 → 테스트 평가 → 오류 확인**

다음 P1 프로젝트에서는 같은 흐름을 **Fashion-MNIST**에 적용한다.

그때부터는 단순히 코드를 따라 실행하는 데서 한 걸음 더 나아가,

> **어떤 모델을 사용할 것인가? 결과를 어떻게 비교할 것인가? 무엇을 개선할 것인가?**

를 스스로 판단하게 된다.